## Kane's method

When doing the lagrangian mechanics, we have already calculated $\dot{\vec{X}_w}$ and $\dot{\vec{X}_p}$, so we can save some trouble here.

### Kane's method intro

Basically:
$$F_r + F_r^* = 0$$

Where, $F_r$ is the generalized active force, and $F_r^*$ is the generalized inertia force.

The generalized active force is defined as:
$$
F_r = \sum_{i} \vec{F}_i \cdot \frac{\partial \dot{\vec{X}}_i}{\partial \dot{q}_r}
$$

Which in this case includes the gravity force and the control force for the two motors.

The generalized inertia force is defined as:
$$
F_r^* = -\sum_{i} m_i \ddot{\vec{X}}_i \cdot \frac{\partial \dot{\vec{X}}_i}{\partial \dot{q}_r}
$$ 

Intrinsically, the generalized inertia force is $\sum m_i {a}_i$.

Let $q = [q_1, q_2, q_3, q_4, q_5]^T = [\psi, \theta, \phi, r, \gamma]^T$ be the vector of generalized coordinates.

I just found that the position $\vec{X}_w$ is not correct:

$$
\vec{X}_w = R \vec{z}_2 + R q_3 \vec{x}_1
$$ 
is not reasonable because the change of direction of $\vec{x}_1$ will change the position of the wheel, making it not longer $R q_3$ away from the origin. 

To calculate correctly, we can use
$$
\vec{X}_w = x(t) \vec{x}_0 + y(t) \vec{y}_0 + R \vec{z}_2
$$

Assume sliding without slipping, we have
$$\dot{x} = R \dot{q}_3 \cos q_1
$$
$$\dot{y} = R \dot{q}_3 \sin q_1
$$

Luckily we have python library that can do most of the work for us, with given $\vec{X}_i$ and $q_r$. As shown below:

In [9]:
from sympy.physics.mechanics import ReferenceFrame, Point, dynamicsymbols, KanesMethod, Particle, RigidBody
from sympy import init_printing, simplify
from Parameters import params
from IPython.display import display

R = params.R
h = params.h

# Generalized coordinates and speeds
q1, q2, q3, q4, q5 = [dynamicsymbols(r'\psi'), dynamicsymbols(r'\theta'), dynamicsymbols(r'\phi'), dynamicsymbols('r'), dynamicsymbols(r'\gamma')]# psi, theta, phi, rho, gamma
u1, u2, u3, u4, u5 = [dynamicsymbols(r'\dot{\psi}'), dynamicsymbols(r'\dot{\theta}'), dynamicsymbols(r'\dot{\phi}'), dynamicsymbols(r'\dot{r}'), dynamicsymbols(r'\dot{\gamma}')]# corresponding generalized speeds

kde = {q1.diff(): u1, q2.diff(): u2, q3.diff(): u3, q4.diff(): u4, q5.diff(): u5} # type: ignore

E0 = ReferenceFrame('E0')
E1 = E0.orientnew('E1', 'Axis', [q1, E0.z]) # rotate around z-axis by psi
E2 = E1.orientnew('E2', 'Axis', [q2, E1.x]) # rotate around x-axis by theta
E3 = E2.orientnew('E3', 'Axis', [q5, E2.y]) # rotate around y-axis by gamma

O = Point('O') # origin
O.set_vel(E0, 0) # origin is fixed

################################### Wheel center kinematics ###################################

# As X_w = x(t) x_0 + y(t) y_0 + R z_2, we can express the position of the wheel center in terms of the initial position X_w0 = R z_2 and consider the contribution of derivatives, which are the speeds.

W = Point('W') # wheel center
W.set_pos(O, R*E2.z) # type: ignore     Initial position

W.set_vel(E0, R * u3 * E1.x + W.pos_from(O).dt(E0).subs(kde)) # type: ignore     velocity, dt(E0) means the time derivative in the inertial frame E0

################################## Pendulum kinematics ########################################

P = W.locatenew('P', h * E3.z) # create Pendulum relative to the wheel
P.v2pt_theory(W, E0, E3)

################################## Linear motor kinematics ########################################

L = W.locatenew('L', (q4) * E2.y)
L.set_vel(E2, u4 * E2.y) # relative velocity in E2 frame, which is the velocity of the linear motor relative to the wheel center
L.v2pt_theory(W, E0, E2)


############################# Define particles and forces ########################################
body_p = Particle('body_p', P, params.m)
body_w = Particle('body_w', W, params.m_w)
body_l = Particle('body_l', L, params.m_l)

#### Forces

T_W = dynamicsymbols('T_W') # torque applied to the wheel
F_L = dynamicsymbols('F_L') # force applied to the linear motor


g = params.g
forces = [
    (P, -params.m * g * E0.z),
    (W, -params.m_w * g * E0.z),
    (L, -params.m_l * g * E0.z),

    ## Wheel torque
    (E3, T_W * E2.y),    # torque applied to the wheel, acting in the E3 frame along the y-axis, which is the axis of rotation for the wheel
    (E2, -T_W * E2.y),   # type: ignore    
    ## Linear motor force
    (L, F_L * E2.y),# linear motor force, acting in the E2 frame along the y-axis
    (W, - F_L * E2.y) # type: ignore     reaction force on the wheel center due to the linear motor, acting in the opposite direction of the force on the linear motor
]

########################### Summarize the system and derive equations of motion ############################

particles = [body_p, body_w, body_l]
q_list = [q1, q2, q3, q4, q5]
u_list = [u1, u2, u3, u4, u5]

kd_eqs = [u1 - q1.diff(), u2 - q2.diff(), u3 - q3.diff(), u4 - q4.diff(), u5 - q5.diff()] # type: ignore

kane = KanesMethod(E0, q_ind=q_list, u_ind=u_list, kd_eqs=kd_eqs) # on interial frame E0

(fr, frstar) = kane.kanes_equations(particles, forces) # Calculate the generalized active forces and inertia forces

mm = kane.mass_matrix_full
ff = kane.forcing_full


########################## Output ###########################

## M(q) * u' = f(q, u, t)

init_printing(use_latex='mathjax')
display(simplify(mm))
display(simplify(ff))

display(simplify(fr))







⎡1  0  0  0  0                                                                 ↪
⎢                                                                              ↪
⎢0  1  0  0  0                                                                 ↪
⎢                                                                              ↪
⎢0  0  1  0  0                                                                 ↪
⎢                                                                              ↪
⎢0  0  0  1  0                                                                 ↪
⎢                                                                              ↪
⎢0  0  0  0  1                                                                 ↪
⎢                                                                              ↪
⎢                                                              2           2   ↪
⎢0  0  0  0  0  0.5⋅(r(t)⋅cos(\theta(t)) - 0.25⋅sin(\theta(t)))  + 0.01⋅sin (\ ↪
⎢                           

⎡                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                                                                              ↪
⎢                   2                                                    2     ↪
⎢0.0125⋅\dot{\gamma} (t)⋅cos(\gamma(t) - \theta(t)) - 0.0125⋅\dot{\gamma} (t)⋅ ↪
⎢                           

⎡                                           0                                  ↪
⎢                                                                              ↪
⎢-4.905⋅r(t)⋅cos(\theta(t)) + 0.981⋅sin(\theta(t))⋅cos(\gamma(t)) + 7.3575⋅sin ↪
⎢                                                                              ↪
⎢                                           0                                  ↪
⎢                                                                              ↪
⎢                                           0                                  ↪
⎢                                                                              ↪
⎣                      T_W(t) + 0.981⋅sin(\gamma(t))⋅cos(\theta(t))            ↪

↪            ⎤
↪            ⎥
↪ (\theta(t))⎥
↪            ⎥
↪            ⎥
↪            ⎥
↪            ⎥
↪            ⎥
↪            ⎦